In [1]:
num_targets = 10
num_particles = 150
state_vector_size = 4  # x,y,h,s
drone_state_input_size = 5  # x y psi budget variance
input_size = num_targets * num_particles * state_vector_size + drone_state_input_size

In [2]:
lightning_ckpt_file = "/home/andrew/Development/onr/onr_ws/src/ipp_planners/neuralnet/lightning_logs/2022-09-16_mean-explicit-rimcts/version_1/checkpoints/best-val-epoch=509-val_loss=4.4120E-04.ckpt"
output_file_name = "limcts_explicit-mean_epoch=509-val_loss=4.4120E-04.pt"

In [3]:
import os

In [4]:
import torch
import pytorch_lightning as pl

In [5]:
from train import ValueEstimator

In [6]:
checkpoint = torch.load(lightning_ckpt_file)

In [7]:
print(checkpoint.keys())

dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers'])


In [8]:
from networks import LinearModel

In [9]:
model = LinearModel()

In [10]:
my_state_dict = checkpoint["state_dict"]

In [11]:
from collections import OrderedDict

new_state_dict = OrderedDict()

bad_keys = my_state_dict.keys()
for key in my_state_dict.keys():
    new_key = key.replace("model.", "")
    new_state_dict[new_key] = my_state_dict[key]

In [12]:
model.load_state_dict(new_state_dict)
model.eval()

LinearModel(
  (layer_a): Linear(in_features=6005, out_features=6000, bias=True)
  (layer_ab): BatchNorm1d(6000, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer_ar): ReLU()
  (layer_ad): Dropout(p=0.5, inplace=False)
  (layer_b): Linear(in_features=6000, out_features=6000, bias=True)
  (layer_bb): BatchNorm1d(6000, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer_br): ReLU()
  (layer_bd): Dropout(p=0.5, inplace=False)
  (layer_c): Linear(in_features=6000, out_features=4000, bias=True)
  (layer_cb): BatchNorm1d(4000, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer_cr): ReLU()
  (layer_cd): Dropout(p=0.5, inplace=False)
  (layer_d): Linear(in_features=4000, out_features=4000, bias=True)
  (layer_db): BatchNorm1d(4000, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer_dr): ReLU()
  (layer_dd): Dropout(p=0.5, inplace=False)
  (layer_e): Linear(in_features=4000, out_features=2000, bias=True)
  (

In [13]:
sample_particle_input = torch.rand(1, num_targets*num_particles*state_vector_size)
sample_drone_input = torch.rand(1, drone_state_input_size)

In [14]:
traced_script_module = torch.jit.trace(model, (sample_particle_input, sample_drone_input))

In [15]:
traced_script_module(sample_particle_input, sample_drone_input)

tensor([[-0.1270]], grad_fn=<AddmmBackward0>)

In [16]:
model(sample_particle_input, sample_drone_input)

tensor([[-0.1270]], grad_fn=<AddmmBackward0>)

In [17]:
traced_script_module.save(output_file_name)
print("Done! Output to:", output_file_name)

Done! Output to: limcts_explicit-mean_epoch=509-val_loss=4.4120E-04.pt
